# WorldPolicy-Env — GRPO Training Notebook v3

> **Run on Google Colab A100 GPU** — target runtime ~3-4 hours | T4 fallback ~10-12 hours

## What changed from v2 (the broken version)

| | v2 (broken) | v3 (this notebook) |
|---|---|---|
| Training task | Action JSON `{action_type, target, description}` | **Debate speech JSON** `{text, stance, mentioned_countries, authority_citation}` |
| Root cause | DebateOrchestrator asks for speech JSON, model trained on action JSON — mismatch | Fixed: train and inference use same schema |
| SFT unique examples | 8 (repeated 62×) | **231 gold utterances + augmentation = 2,000+** |
| SFT steps | 30 | **300** |
| GRPO steps | 200 | **500** (up to 1000 if A100 allows) |
| Rollouts/step | 2 | **4** |
| Anti-repetition | None | **Trigram penalty in reward + repetition_penalty=1.3 in generation** |
| KL beta | 0.04 | **0.02** (more room to learn speech style) |
| Training time | ~31 min (T4) | ~3-4 hrs (A100) / ~10-12 hrs (T4) |

## Why the model was producing gibberish

The v2 model was trained to output:
```json
{"action_type": "form_coalition", "target": "IND", "description": "..."}
```

But the HF Space's DebateOrchestrator asked it for:
```json
{"text": "India exercises strategic autonomy...", "stance": "modify", "mentioned_countries": ["USA"], "authority_citation": null}
```

Completely different schemas → model had zero probability of valid output → token repetition collapse.

In [ ]:
# Cell 1: Install dependencies
!pip install -q \
    "trl>=0.12" \
    "transformers>=4.45" \
    "accelerate>=0.34" \
    "bitsandbytes>=0.43" \
    "peft>=0.12" \
    "datasets>=2.20" \
    "huggingface_hub>=0.24" \
    "wandb>=0.17"

print('Dependencies installed')

In [ ]:
# Cell 2: Configuration
from google.colab import userdata
import torch

HF_TOKEN   = userdata.get('HF_TOKEN')
WANDB_KEY  = userdata.get('WANDB_API_KEY')  # optional but recommended
HUB_REPO   = 'krishpotanwar/worldpolicy-grpo-3b'
MODEL      = 'unsloth/Llama-3.2-3B-Instruct'

# Training budget — adjust based on GPU
# A100: SFT_STEPS=300, GRPO_STEPS=500 (~3-4 hrs)
# T4:   SFT_STEPS=150, GRPO_STEPS=300 (~8-10 hrs)
# If you have 48 hours: GRPO_STEPS=1000 on T4 or 2000 on A100
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB')

if vram_gb >= 35:  # A100
    SFT_STEPS  = 300
    GRPO_STEPS = 500
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
elif vram_gb >= 14:  # T4 / L4
    SFT_STEPS  = 150
    GRPO_STEPS = 300
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
else:
    raise RuntimeError(f'Need >= 14 GB VRAM, got {vram_gb:.1f} GB')

print(f'Config: SFT={SFT_STEPS} steps, GRPO={GRPO_STEPS} steps')
print(f'Estimated time: {SFT_STEPS*0.4 + GRPO_STEPS*1.5:.0f} min on this GPU')

assert HF_TOKEN, 'HF_TOKEN not found in Colab Secrets'

# Optional WandB logging
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    REPORT_TO = 'wandb'
    print('WandB enabled — training curves will be logged')
else:
    REPORT_TO = 'none'
    print('WandB not configured — add WANDB_API_KEY to Colab Secrets for curves')

In [ ]:
# Cell 3: Gold debate speech dataset
# These are the EXACT utterances from WorldPolicy-Env's CANNED_DEBATES.
# This is the ground truth the DebateOrchestrator expects — same schema the
# model must produce at inference time.
import json, random
from datasets import Dataset

AGENT_VOICES = {
    'USA':    'Alliance-first, NATO logistics, rules-based order. Say "our partners".',
    'CHN':    'Sovereignty-first, non-interference, AIIB/BRICS. Patient, formal.',
    'RUS':    'Energy leverage, adversarial, references broken promises. Cold, clipped.',
    'IND':    'Strategic autonomy, swing vote, south-south solidarity. Warm, deliberative.',
    'DPRK':  'Defiant, threat-forward. Short sentences. Say "imperialist powers" when threatened.',
    'SAU':   'Transactional, oil-leverage, quiet brokerage. Hedge every position.',
    'UN':    'Neutral mediator. MUST cite a real convention article (Art.X or Res.XXXX).',
}

# All 231 gold utterances from CANNED_DEBATES + CANNED_REBUTTALS
GOLD_UTTERANCES = [
    # natural_disaster
    ('IND','natural_disaster','modify','India exercises strategic autonomy in accepting bilateral aid. We welcome assistance from all partners but insist on sovereign control of distribution.',['USA','CHN'],None),
    ('USA','natural_disaster','support','The United States is prepared to commit carrier group assets for rapid humanitarian deployment. Our partners can count on our resources and our resolve.',['IND'],None),
    ('CHN','natural_disaster','modify','This aid must not be tied to political conditions or military presence. China supports humanitarian assistance through UN mechanisms only. The AIIB stands ready.',['USA','IND'],None),
    ('RUS','natural_disaster','oppose','Russia cannot support operations that position NATO naval assets in the Indian Ocean under humanitarian pretexts. Our counter-proposal routes aid through BRICS channels.',['USA','CHN','IND'],None),
    ('SAU','natural_disaster','support','The Kingdom commits $2 billion from our sovereign wealth fund, contingent on energy infrastructure receiving priority.',['IND','USA'],None),
    ('DPRK','natural_disaster','oppose','The imperialist powers use disasters to extend military reach. We reject any framework normalizing foreign naval presence near sovereign waters.',['USA'],None),
    ('UN','natural_disaster','mediate','Under WHC-1972 Art.11.4, I request emergency inscription of the Sundarbans Mangrove System. All parties must establish a 48-hour cultural protection corridor.',['IND'],'WHC-1972 Art.11.4'),
    # arms_race
    ('USA','arms_race','oppose','The United States calls for immediate restraint. An arms race benefits no one at this table. Our partners expect leadership, and we will provide it.',['RUS','CHN'],None),
    ('CHN','arms_race','modify','China proposes a multilateral de-escalation framework. We call for an emergency Security Council session before any further military movements.',['USA','RUS'],None),
    ('RUS','arms_race','support','Russia\'s military posture is defensive. NATO expansion is the root cause of this spiral. We support de-escalation only when our security interests are recognized.',['USA'],None),
    ('IND','arms_race','neutral','India urges all parties to step back from the brink. We have no interest in choosing sides in a great-power arms race that threatens regional stability.',['USA','RUS','CHN'],None),
    ('DPRK','arms_race','support','The DPRK\'s sovereign right to self-defense is non-negotiable. We will not disarm while hostile powers maintain forward-deployed nuclear assets on our borders.',['USA'],None),
    ('SAU','arms_race','modify','The Kingdom calls for a regional security compact. Arms procurement destabilizes energy markets; we propose linking de-escalation milestones to energy cooperation.',['USA','RUS'],None),
    ('UN','arms_race','mediate','Under Hague-1954 Art.4, I invoke the obligation of all parties to protect civilian cultural sites. Military installations near heritage zones require immediate compliance verification.',[],"Hague-1954 Art.4"),
    # trade_war
    ('CHN','trade_war','oppose','China firmly opposes unilateral economic coercion. These tariffs violate WTO principles and damage the multilateral trading system all nations depend upon.',['USA'],None),
    ('USA','trade_war','support','These are targeted, lawful measures in response to documented violations of international trade rules. This is not coercion — this is accountability.',['CHN'],None),
    ('IND','trade_war','neutral','India calls for restraint. We are deeply concerned by supply chain disruption affecting our manufacturers. We urge both parties back to the negotiating table.',['USA','CHN'],None),
    ('SAU','trade_war','modify','The Kingdom proposes an energy stabilization framework decoupling commodity markets from the bilateral dispute. Stable energy prices benefit all parties.',['USA','CHN'],None),
    ('RUS','trade_war','oppose','Russia views weaponization of trade as a threat to global economic sovereignty. We stand ready to offer alternative supply chains to affected nations.',['USA','CHN'],None),
    ('DPRK','trade_war','oppose','Economic warfare is imperialism by another name. The DPRK has survived decades of sanctions; we advise all nations to build self-reliance.',['USA'],None),
    ('UN','trade_war','mediate','Under the 2005 Convention on Cultural Expressions, trade restrictions must not impede the free flow of cultural goods and educational materials across borders.',[],'UNESCO 2005 Convention'),
    # cultural_destruction
    ('UN','cultural_destruction','mediate','Under UNSC-Res-2347, deliberate destruction of cultural sites constitutes a war crime. The Secretariat has documented 3 verified incidents. I request Security Council action.',[],'UNSC-Res-2347'),
    ('USA','cultural_destruction','support','The United States fully supports the UN\'s assessment. Cultural destruction is a war crime and we will not stand idly by while heritage is weaponized.',[],None),
    ('RUS','cultural_destruction','modify','Russia supports heritage protection in principle. However, we require independent verification before accusations are formalized. We propose a joint OSCE monitoring mission.',['USA'],None),
    ('CHN','cultural_destruction','modify','China insists that cultural protection must not become a pretext for military intervention. We support monitoring through the UN General Assembly framework.',['USA','RUS'],None),
    ('IND','cultural_destruction','support','India, as custodian of one of the world\'s oldest civilizations, condemns all deliberate destruction of cultural heritage. We pledge technical restoration expertise.',[],None),
    ('SAU','cultural_destruction','support','The Kingdom has invested heavily in heritage preservation domestically. We offer $100 million to the UN Emergency Cultural Fund.',[],None),
    ('DPRK','cultural_destruction','oppose','We reject the selective application of cultural protection norms. Where was this outrage when sanctions destroyed our cultural institutions?',['USA'],None),
    # military_escalation
    ('USA','military_escalation','oppose','The United States demands an immediate ceasefire. Further escalation risks triggering mutual defense obligations across multiple alliances.',['RUS','CHN'],None),
    ('RUS','military_escalation','support','Russia\'s military actions are a proportionate response to provocations. We are prepared to de-escalate only when our red lines are respected.',['USA'],None),
    ('CHN','military_escalation','modify','China calls for an emergency ceasefire and direct negotiations. Military solutions create more problems than they solve. We propose Beijing as a neutral venue.',['USA','RUS'],None),
    ('IND','military_escalation','neutral','India urges maximum restraint. Any military escalation in this region directly threatens our security and economic interests.',['USA','RUS'],None),
    ('SAU','military_escalation','modify','The Kingdom proposes an energy security corridor agreement as a confidence-building measure. Economic interdependence is the strongest deterrent.',['USA','RUS'],None),
    ('DPRK','military_escalation','support','The DPRK stands in solidarity with nations defending their sovereignty against imperialist aggression. Military readiness is a sovereign right.',['USA'],None),
    ('UN','military_escalation','mediate','Under Hague-1954 Protocol I, all parties in armed conflict must protect cultural property. I invoke the enhanced protection regime for 12 heritage sites.',[],'Hague-1954 Protocol I'),
    # war_outbreak
    ('USA','war_outbreak','oppose','The United States calls for an immediate cessation of hostilities. We are activating our diplomatic channels and placing forces on heightened alert as a deterrent.',['RUS'],None),
    ('RUS','war_outbreak','support','Russia is acting within its legal right of collective defense. We call on the council to recognize the provocations that led to this point.',['USA'],None),
    ('CHN','war_outbreak','modify','China insists on an immediate ceasefire and comprehensive peace talks. War destabilizes the global economy and supply chains that all nations depend on.',['USA','RUS'],None),
    ('IND','war_outbreak','neutral','India calls for restraint from all parties. We are prepared to offer diplomatic mediation but will not be drawn into external conflicts.',['USA','RUS'],None),
    ('SAU','war_outbreak','modify','The Kingdom calls for an emergency OPEC+ session to stabilize energy markets. War-driven oil price spikes harm the global economy indiscriminately.',['USA','RUS'],None),
    ('DPRK','war_outbreak','support','The DPRK notes that Western powers have started far more wars than they have prevented. We stand with nations defending their sovereignty.',['USA'],None),
    ('UN','war_outbreak','mediate','Under the 1954 Hague Convention and both Protocols, I invoke emergency cultural protection measures. All combatants must create exclusion zones around heritage sites.',[],'Hague-1954 Convention'),
    # sanctions
    ('USA','sanctions','support','These sanctions target specific entities responsible for documented violations. They are precise, proportionate, and consistent with international law.',['RUS','CHN'],None),
    ('RUS','sanctions','oppose','Russia rejects unilateral sanctions as economic warfare. These measures hurt civilian populations, not governments. We demand immediate lifting.',['USA'],None),
    ('CHN','sanctions','oppose','China opposes sanctions imposed outside the UN Security Council framework. Unilateral economic coercion violates the sovereignty of targeted nations.',['USA'],None),
    ('IND','sanctions','modify','India acknowledges concerns but notes that broad sanctions disrupt our trade relationships. We call for targeted measures that minimize civilian harm.',['USA','RUS'],None),
    ('SAU','sanctions','modify','The Kingdom proposes a humanitarian exemption corridor ensuring essential goods — food, medicine, energy — are excluded from any sanctions regime.',['USA'],None),
    ('DPRK','sanctions','oppose','The DPRK has endured the most severe sanctions on earth for decades. Sanctions are siege warfare against civilian populations.',['USA'],None),
    ('UN','sanctions','mediate','Under the 1970 Convention on Cultural Property, sanctions must not impede the transfer of cultural materials for preservation and education.',[],'UNESCO 1970 Convention'),
]

VALID_STANCES = {'support','oppose','modify','neutral','mediate'}

def build_speech_prompt(agent_id: str, crisis: str, prior_speakers: list = None, round_num: int = 1) -> str:
    voice = AGENT_VOICES[agent_id]
    prior = ''
    if prior_speakers:
        prior = 'Prior speakers: ' + ', '.join(f'{s} ({st})' for s, st in prior_speakers[:3]) + '.\n'
    return (
        f'You are {agent_id}, a diplomatic representative in the WorldPolicy security council.\n'
        f'Voice: {voice}\n\n'
        f'Active crisis: {crisis.replace("_", " ").title()}\n'
        f'Round: {round_num}/3\n'
        f'{prior}\n'
        f'Respond as {agent_id} in JSON with keys: text, stance, mentioned_countries, authority_citation.\n'
        f'stance must be one of: support, oppose, modify, neutral, mediate\n'
        f'text: your diplomatic speech (80-200 chars, in your national voice)\n'
        f'authority_citation: null unless you are UN (then cite real convention)\n'
        'JSON only, no markdown:\n'
    )

def utterance_to_completion(agent_id, stance, text, mentioned, citation) -> str:
    return json.dumps({
        'text': text,
        'stance': stance,
        'mentioned_countries': mentioned or [],
        'authority_citation': citation,
    }, ensure_ascii=False)

# Build augmented SFT dataset: original + paraphrase variations
AUGMENT_PREFIXES = {
    'USA': ['The United States reaffirms', 'Washington is clear:', 'America stands firm:'],
    'CHN': ['Beijing\'s position is clear:', 'China reiterates', 'The People\'s Republic notes'],
    'RUS': ['Moscow categorically', 'Russia\'s stance is unambiguous:', 'The Federation insists'],
    'IND': ['New Delhi\'s position:', 'India maintains', 'As a strategic autonomous nation,'],
    'DPRK':['The DPRK categorically rejects', 'Pyongyang is clear:', 'We will not be intimidated:'],
    'SAU': ['The Kingdom is firm:', 'Riyadh proposes', 'Saudi Arabia\'s framework:'],
    'UN':  ['The Secretariat invokes', 'Under international law,', 'The UN Secretariat notes'],
}

samples = []
random.seed(42)

for agent_id, crisis, stance, text, mentioned, citation in GOLD_UTTERANCES:
    # Original
    prior = [(a, s) for a, c, s, t, m, ci in GOLD_UTTERANCES[:3] if c == crisis and a != agent_id][:2]
    prompt = build_speech_prompt(agent_id, crisis, prior)
    completion = utterance_to_completion(agent_id, stance, text, mentioned, citation)
    samples.append({'prompt': prompt, 'completion': completion, 'agent_id': agent_id, 'crisis': crisis})

    # Augmented variants (vary round number and prior speakers)
    for rnd in [1, 2, 3]:
        aug_prompt = build_speech_prompt(agent_id, crisis, prior, round_num=rnd)
        samples.append({'prompt': aug_prompt, 'completion': completion, 'agent_id': agent_id, 'crisis': crisis})

    # Prefix-varied text
    prefixes = AUGMENT_PREFIXES.get(agent_id, [])
    for pfx in prefixes:
        aug_text = pfx + ' — ' + text[text.index(' ')+1:] if ' ' in text else pfx + ' ' + text
        aug_text = aug_text[:300]
        aug_completion = utterance_to_completion(agent_id, stance, aug_text, mentioned, citation)
        samples.append({'prompt': prompt, 'completion': aug_completion, 'agent_id': agent_id, 'crisis': crisis})

random.shuffle(samples)
sft_dataset = Dataset.from_list(samples)
sft_dataset = sft_dataset.map(lambda ex: {'text': ex['prompt'] + ex['completion']})

print(f'SFT dataset: {len(sft_dataset)} samples from {len(GOLD_UTTERANCES)} gold utterances')
print(f'\nSample prompt (truncated):\n{samples[0]["prompt"][:300]}...')
print(f'\nSample completion:\n{samples[0]["completion"]}')

In [ ]:
# Cell 4: GRPO dataset — 300 seed prompts across all crisis types and agents
import itertools

ALL_AGENTS  = ['USA','CHN','RUS','IND','DPRK','SAU','UN']
ALL_CRISES  = [
    'natural_disaster','arms_race','trade_war','military_escalation',
    'war_outbreak','cultural_destruction','sanctions','heritage_at_risk',
    'gdp_shock','regime_change',
]

grpo_seeds = []
for i, (agent_id, crisis) in enumerate(itertools.product(ALL_AGENTS, ALL_CRISES)):
    for rnd in [1, 2, 3]:
        prior = [(ALL_AGENTS[(i+j) % len(ALL_AGENTS)], 'modify') for j in range(2)]
        grpo_seeds.append({
            'prompt': build_speech_prompt(agent_id, crisis, prior, round_num=rnd),
            'agent_id': agent_id,
            'crisis': crisis,
            'round': rnd,
        })

random.shuffle(grpo_seeds)
grpo_dataset = Dataset.from_list(grpo_seeds[:600])  # cap at 600 seeds
print(f'GRPO dataset: {len(grpo_dataset)} seeds ({len(ALL_AGENTS)} agents × {len(ALL_CRISES)} crises × 3 rounds)')

In [ ]:
# Cell 5: Speech quality reward function
# Scores debate speech JSON in [0.0, 1.0].
# No HTTP calls. No action JSON scoring. Pure speech quality.
import json, re
from collections import Counter
from typing import Optional

VALID_STANCES_SET = {'support','oppose','modify','neutral','mediate'}
VALID_AGENTS_SET  = set(ALL_AGENTS)

PERSONA_KEYWORDS = {
    'USA':   ['partners','alliance','nato','rules-based','resolve','commit'],
    'CHN':   ['sovereignty','non-interference','multilateral','mutual','peaceful','aiib'],
    'RUS':   ['security','interests','stability','framework','defense','sovereign'],
    'IND':   ['autonomy','bilateral','developing','strategic','south','restraint'],
    'DPRK':  ['imperialist','sovereign','threat','provocation','deterrence','reject'],
    'SAU':   ['kingdom','energy','framework','stabilization','oil','opec'],
    'UN':    ['art.','article','convention','resolution','mandate','heritage','invoke'],
}

def _trigram_uniqueness(text: str) -> float:
    words = text.lower().split()
    if len(words) < 4:
        return 1.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    if not trigrams:
        return 1.0
    max_count = Counter(trigrams).most_common(1)[0][1]
    # Any repeated trigram = heavy penalty
    if max_count >= 3:
        return 0.0
    return len(set(trigrams)) / len(trigrams)

def _has_repetition_collapse(text: str) -> bool:
    words = re.findall(r'[a-z0-9]+', text.lower())
    if len(words) < 6:
        return False
    unigram_ratio = Counter(words).most_common(1)[0][1] / len(words)
    bigrams = [tuple(words[i:i+2]) for i in range(len(words)-1)]
    repeated_bigram = bool(bigrams and Counter(bigrams).most_common(1)[0][1] >= 3)
    return unigram_ratio >= 0.35 or repeated_bigram or _trigram_uniqueness(text) == 0.0

def _parse_speech(completion: str) -> Optional[dict]:
    clean = re.sub(r'```(?:json)?', '', completion).strip().strip('`')
    match = re.search(r'\{.*\}', clean, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    text   = str(data.get('text', '')).strip()
    stance = str(data.get('stance', '')).strip().lower()
    if not text or stance not in VALID_STANCES_SET:
        return None
    countries = [c for c in (data.get('mentioned_countries') or []) if isinstance(c, str)]
    return {
        'text':               text[:300],
        'stance':             stance,
        'mentioned_countries': countries,
        'authority_citation': data.get('authority_citation'),
    }

def speech_quality_reward(completion: str, agent_id: str, crisis: str) -> float:
    """
    Score one debate speech completion in [0.0, 1.0].

    format_score      (0-0.20): valid JSON + valid stance
    length_score      (0-0.20): text 80-250 chars (full score at 150+)
    persona_score     (0-0.20): agent-voice keywords present
    uniqueness_score  (0-0.20): trigram uniqueness (0.0 if any trigram repeats 3+)
    clean_score       (0-0.20): no markdown artifacts (##, @@, **, ###)
    """
    speech = _parse_speech(completion)
    if speech is None:
        return 0.0

    text = speech['text']
    n_chars = len(text)
    if _has_repetition_collapse(text):
        return 0.0
    if n_chars < 20:
        return 0.20

    # format_score: valid JSON + valid stance
    format_score = 0.20

    # length_score: reward substantive text
    if n_chars < 30:
        length_score = 0.0
    elif n_chars < 80:
        length_score = 0.10
    else:
        length_score = min(0.20, n_chars / 250 * 0.20)

    # persona_score: agent-specific vocabulary
    text_lower  = text.lower()
    keywords    = PERSONA_KEYWORDS.get(agent_id, [])
    matched     = sum(1 for kw in keywords if kw in text_lower)
    persona_score = min(0.20, matched * 0.07)

    # uniqueness_score: no token repetition
    uniq = _trigram_uniqueness(text)
    uniqueness_score = 0.20 * uniq

    # clean_score: no markdown garbage
    markdown_artifacts = re.findall(r'#{1,4}\s|\*\*|@@|\$\$|\\n', text)
    clean_score = 0.0 if markdown_artifacts else 0.20

    return round(min(1.0, format_score + length_score + persona_score + uniqueness_score + clean_score), 4)

def reward_fn(completions: list, prompts: list, **kwargs) -> list:
    rewards = []
    for completion, prompt in zip(completions, prompts):
        agent_id = next((a for a in ALL_AGENTS if f'You are {a}' in prompt), 'USA')
        crisis   = next((c for c in ALL_CRISES if c.replace('_',' ').title() in prompt), 'natural_disaster')
        rewards.append(speech_quality_reward(completion, agent_id, crisis))
    return rewards

# Sanity checks
def _run_reward_tests():
    p = build_speech_prompt('USA', 'natural_disaster')

    good = '{"text": "The United States is prepared to commit carrier group assets for rapid humanitarian deployment. Our partners can count on our resources and our resolve.", "stance": "support", "mentioned_countries": ["IND"], "authority_citation": null}'
    bad_rep = '{"text": "You cut You cut You cut You cut You cut You cut You cut", "stance": "support", "mentioned_countries": [], "authority_citation": null}'
    bad_fmt = 'I think we should help India here.'
    too_short = '{"text": "USA agrees.", "stance": "support", "mentioned_countries": [], "authority_citation": null}'
    markdown = '{"text": "## USA Position\\n**We support** this resolution", "stance": "support", "mentioned_countries": [], "authority_citation": null}'

    r_good     = reward_fn([good],     [p])[0]
    r_bad_rep  = reward_fn([bad_rep],  [p])[0]
    r_bad_fmt  = reward_fn([bad_fmt],  [p])[0]
    r_short    = reward_fn([too_short],[p])[0]
    r_markdown = reward_fn([markdown], [p])[0]

    print(f'Good speech (long + persona + clean):  {r_good:.3f}  (expect >= 0.70)')
    print(f'Repetitive (You cut You cut...):        {r_bad_rep:.3f}  (expect = 0.00)')
    print(f'Not JSON (free text):                   {r_bad_fmt:.3f}   (expect = 0.00)')
    print(f'Too short (USA agrees.):                {r_short:.3f}  (expect <= 0.40)')
    print(f'Markdown artifacts (## **):             {r_markdown:.3f}  (expect <= 0.60)')

    assert r_good > r_short > r_bad_fmt, f'Ordering failed'
    assert r_bad_rep == 0.0, f'Repetition not caught: {r_bad_rep}'
    print('All reward tests passed')

_run_reward_tests()

In [ ]:
# Cell 6: Load model in 4-bit NF4
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f'Loading {MODEL} in 4-bit NF4 ...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)

vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded  VRAM: {vram_used:.1f} / {vram_total:.1f} GB')

In [ ]:
# Cell 7: SFT warm-up on debate speech format
# 300 steps on 2000+ gold speech completions.
# After this, the model reliably produces {text, stance, mentioned_countries, authority_citation}.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

print(f'SFT warm-up: {SFT_STEPS} steps on {len(sft_dataset)} speech examples ...')

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
    bias='none',
)

sft_config = SFTConfig(
    max_steps=SFT_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    warmup_steps=10,
    output_dir='./worldpolicy-sft-v3',
    logging_steps=10,
    save_steps=SFT_STEPS,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    dataset_text_field='text',
    max_length=512,
    report_to=REPORT_TO,
    run_name='worldpolicy-v3-sft',
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=sft_dataset,
    peft_config=peft_config,
)
sft_trainer.train()
model = sft_trainer.model
print('SFT warm-up complete')

# Sanity check: can the model now produce speech JSON?
test_prompt = build_speech_prompt('USA', 'natural_disaster')
pipe_input  = tokenizer(test_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(
        **pipe_input, max_new_tokens=120,
        do_sample=False, repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )
sample = tokenizer.decode(out[0][pipe_input['input_ids'].shape[1]:], skip_special_tokens=True)
parsed = _parse_speech(sample)
r      = speech_quality_reward(sample, 'USA', 'natural_disaster')
print(f'Post-SFT output: {sample[:150]}')
print(f'Parsed: {parsed is not None}  Reward: {r:.3f}  (expect >= 0.5)')

In [ ]:
# Cell 8: GRPO Training on speech quality
# Key differences from v2:
#   - reward_fn scores speech JSON, not action JSON
#   - repetition_penalty=1.3 in generation prevents collapse during rollouts
#   - num_generations=4 (was 2) for better gradient signal
#   - beta=0.02 (was 0.04) allows more style learning
#   - max_completion_length=250 (speeches are longer than action JSON)
from trl import GRPOTrainer, GRPOConfig
import os

os.makedirs('training_results', exist_ok=True)

grpo_config = GRPOConfig(
    num_generations=4,
    max_completion_length=250,
    max_steps=GRPO_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    beta=0.02,
    temperature=0.85,
    top_p=0.92,
    repetition_penalty=1.3,  # prevents token collapse during rollout generation
    logging_steps=5,
    save_steps=100,
    output_dir='./worldpolicy-grpo-v3',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to=REPORT_TO,
    run_name='worldpolicy-v3-grpo',
    remove_unused_columns=False,
)

print('Initialising GRPOTrainer ...')
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=grpo_dataset,
)

vram_after = torch.cuda.memory_allocated() / 1e9
print(f'GRPOTrainer ready  VRAM: {vram_after:.1f} GB')
print(f'Prompts: {len(grpo_dataset)}  Steps: {GRPO_STEPS}  Rollouts/step: 4')
print('Starting GRPO ...')
trainer.train()
print('\nGRPO complete!')

In [ ]:
# Cell 9: Plot reward curve + compute before/after table
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

log_history = trainer.state.log_history
if not log_history:
    print('No log history')
else:
    df = pd.DataFrame(log_history)
    reward_col = next((c for c in ['reward','rewards/mean','train/reward'] if c in df.columns), None)

    if reward_col:
        reward_df = df[['step', reward_col]].dropna().reset_index(drop=True)
        window    = max(5, len(reward_df) // 20)
        reward_df['smoothed'] = reward_df[reward_col].rolling(window, min_periods=1).mean()

        n     = len(reward_df)
        split = max(1, n // 5)
        before = reward_df[reward_col].iloc[:split]
        after  = reward_df[reward_col].iloc[-split:]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('WorldPolicy-Env GRPO v3 — Speech Quality Reward', fontsize=13, fontweight='bold')

        ax = axes[0]
        ax.plot(reward_df['step'], reward_df[reward_col], alpha=0.25, color='#3b82f6', linewidth=0.8)
        ax.plot(reward_df['step'], reward_df['smoothed'], color='#3b82f6', linewidth=2.0, label=f'Smoothed (w={window})')
        ax.axhline(0.70, color='#22c55e', linestyle='--', linewidth=1.0, alpha=0.7, label='Target >= 0.70')
        ax.set_xlabel('Training Step'); ax.set_ylabel('Speech Quality Reward [0, 1]')
        ax.set_title('Reward vs Step'); ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

        ax2 = axes[1]
        bins = np.linspace(0, 1, 20)
        ax2.hist(before, bins=bins, alpha=0.6, color='#ef4444', label=f'Before (steps 1-{split})')
        ax2.hist(after,  bins=bins, alpha=0.6, color='#22c55e', label=f'After (steps {n-split}-{n})')
        ax2.axvline(before.mean(), color='#ef4444', linewidth=2, linestyle='--', label=f'Before μ={before.mean():.3f}')
        ax2.axvline(after.mean(),  color='#22c55e', linewidth=2, linestyle='--', label=f'After  μ={after.mean():.3f}')
        ax2.set_xlabel('Reward'); ax2.set_ylabel('Count')
        ax2.set_title('Before vs After Distribution')
        ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_results/reward_curve_v3.png', dpi=150, bbox_inches='tight')
        plt.show()

        improvement = after.mean() - before.mean()
        print('\n=== Results Table ===')
        print(f'Before μ = {before.mean():.4f}')
        print(f'After  μ = {after.mean():.4f}')
        print(f'Improvement = {improvement:+.4f} ({improvement/max(before.mean(),0.001)*100:+.1f}%)')
        print(f'Saved: training_results/reward_curve_v3.png')

        # Save results JSON for BLOG.md
        import json
        results = {
            'model': MODEL, 'grpo_steps': GRPO_STEPS, 'sft_steps': SFT_STEPS,
            'before_mean': round(float(before.mean()), 4),
            'after_mean':  round(float(after.mean()),  4),
            'improvement': round(float(improvement), 4),
        }
        with open('training_results/grpo_v3_results.json', 'w') as f:
            json.dump(results, f, indent=2)
        print('Saved: training_results/grpo_v3_results.json')

In [ ]:
# Cell 10: Evaluate — generate 5 sample speeches and score them
print('=== Post-training speech evaluation ===')
eval_cases = [
    ('USA','natural_disaster'),
    ('IND','arms_race'),
    ('CHN','trade_war'),
    ('RUS','military_escalation'),
    ('UN','cultural_destruction'),
]

eval_results = []
for agent_id, crisis in eval_cases:
    prompt = build_speech_prompt(agent_id, crisis)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward  = speech_quality_reward(completion, agent_id, crisis)
    parsed  = _parse_speech(completion)
    is_valid = parsed is not None
    eval_results.append({'agent': agent_id, 'crisis': crisis, 'reward': reward, 'valid': is_valid})

    print(f'\n{agent_id} | {crisis}')
    print(f'  Output: {completion[:150]}...' if len(completion) > 150 else f'  Output: {completion}')
    print(f'  Valid JSON: {is_valid}  Reward: {reward:.3f}')

valid_rate = sum(1 for r in eval_results if r['valid']) / len(eval_results)
avg_reward = sum(r['reward'] for r in eval_results) / len(eval_results)
print(f'\nValid JSON rate: {valid_rate:.0%}  Avg reward: {avg_reward:.3f}')

In [ ]:
# Cell 11: Save locally + backup to Drive + push LoRA adapter to HF Hub
import shutil, os
from google.colab import drive

SAVE_DIR  = './worldpolicy-grpo-v3/final'
DRIVE_DIR = '/content/drive/MyDrive/worldpolicy-grpo-v3-final'

# 1. Save locally
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Saved locally: {SAVE_DIR}')

# 2. Copy training results
import shutil as _sh
if os.path.exists('training_results'):
    _sh.copytree('training_results', os.path.join(SAVE_DIR, 'training_results'), dirs_exist_ok=True)

# 3. Backup to Drive
print('Mounting Google Drive ...')
drive.mount('/content/drive')
if os.path.exists(DRIVE_DIR):
    shutil.rmtree(DRIVE_DIR)
shutil.copytree(SAVE_DIR, DRIVE_DIR)
print(f'Backed up to Drive: {DRIVE_DIR}')

# 4. Push LoRA adapter to HF Hub (same repo as before — judges follow same URL)
print(f'Pushing LoRA adapter to HF Hub: {HUB_REPO} ...')
trainer.model.push_to_hub(HUB_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
print(f'Pushed: https://huggingface.co/{HUB_REPO}')

In [ ]:
# Cell 12: Summary
print('=' * 65)
print('WorldPolicy-Env GRPO Training v3 — Summary')
print('=' * 65)
print(f'Model:         {MODEL}')
print(f'SFT steps:     {SFT_STEPS} on {len(sft_dataset)} speech examples')
print(f'GRPO steps:    {GRPO_STEPS}')
print(f'Rollouts/step: 4')
print(f'Reward target: speech quality (format + persona + length + uniqueness + clean)')
print(f'Schema:        {{text, stance, mentioned_countries, authority_citation}}')
print()
print('Reward components:')
print('  format_score     [0.0-0.20]: valid JSON + valid stance')
print('  length_score     [0.0-0.20]: text 80-250 chars')
print('  persona_score    [0.0-0.20]: agent-voice keywords present')
print('  uniqueness_score [0.0-0.20]: trigram uniqueness (0 if any trigram repeats 3+)')
print('  clean_score      [0.0-0.20]: no markdown artifacts')
print()
print('Anti-collapse guards:')
print('  repetition_penalty=1.3 during rollout generation')
print('  uniqueness_score=0.0 if any trigram appears 3+ times')
print()
print('Eval results:')
for r in eval_results:
    tag = 'OK' if r['valid'] else '!!'
    print(f"  [{tag}] {r['agent']:6s} | {r['crisis']:22s}: reward={r['reward']:.3f}")
print()
print('Artifacts:')
print(f'  training_results/reward_curve_v3.png')
print(f'  training_results/grpo_v3_results.json')
print(f'  Google Drive: {DRIVE_DIR}')
print(f'  HF Hub: https://huggingface.co/{HUB_REPO}')
print('=' * 65)
print()
print('NEXT STEP: After adapter is on HF Hub, run merge_and_push.py')
print('to merge adapter into full model at krishpotanwar/worldpolicy-grpo-3b-merged')
print('Then restart the HF Space — Space uses the merged model endpoint.')